# Chain Ladder with `chainladder`

This notebook uses `pandas` and [`chainladder`](https://chainladder-python.readthedocs.io/).

It reads the supplied synthetic **Loss Incurred** database, converts it to the long format expected by the package, fits volume-weighted Chain Ladder development factors, and produces aggregate actual, projected maturity/ultimate, and reserve totals.

### Step 1 — Import libraries and locate the database

Purpose: import `pandas`, `numpy`, and `chainladder`. Locate the supplied database.

Produces: `input_path`, the path to the supplied CSV.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import chainladder as cl

AMOUNT_FORMAT = '{:,.0f}'
FACTOR_FORMAT = '{:.3f}'

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
input_path = project_root /'generated_monthly_triangles_database.csv'

if not input_path.exists():
    raise FileNotFoundError(f'Database not found: {input_path}')

print(f'Using database: {input_path.name}')
print(f'chainladder version: {cl.__version__}')

Using database: generated_monthly_triangles_database.csv
chainladder version: 0.10.0


### Step 2 — Read and filter the Loss Incurred records

Purpose: read the long-format database and retain only monthly, cumulative Loss Incurred. The package receives one amount per accident-period/development-period cell.

Produces: `loss_incurred_long`, the package input before calendar development dates are added.

In [4]:
database = pd.read_csv(input_path)
required_columns = {
    'concept', 'basis', 'amount_type', 'accident_period',
    'development_period', 'amount',
}
missing = required_columns - set(database.columns)
if missing:
    raise ValueError(f'Database is missing required columns: {sorted(missing)}')

loss_incurred_long = database.loc[
    (database['concept'] == 'Loss Incurred')
    & (database['basis'] == 'month')
    & (database['amount_type'] == 'cumulative'),
    ['accident_period', 'development_period', 'amount'],
].copy()

if loss_incurred_long.empty:
    raise ValueError('No monthly cumulative Loss Incurred records were found.')
if loss_incurred_long.duplicated(['accident_period', 'development_period']).any():
    raise ValueError('Expected one record per accident/development cell after filtering.')

print('Filtered Loss Incurred cells:', len(loss_incurred_long))
print('Accident periods:', loss_incurred_long['accident_period'].nunique())
print('Development ages:', loss_incurred_long['development_period'].nunique())
print('\nDatabase fields used in this demonstration:')
loss_incurred_long.head(10).style.format({'amount': AMOUNT_FORMAT})

Filtered Loss Incurred cells: 7260
Accident periods: 120
Development ages: 120

Database fields used in this demonstration:


,accident_period,development_period,amount
14520,2016-01,0,"153,701"
14521,2016-02,0,"174,526"
14522,2016-03,0,"139,978"
14523,2016-04,0,"179,441"
14524,2016-05,0,"185,553"
14525,2016-06,0,"257,005"
14526,2016-07,0,"221,999"
14527,2016-08,0,"208,672"
14528,2016-09,0,"278,668"
14529,2016-10,0,"188,245"


### Step 3 — Create package calendar development dates

Purpose: `chainladder.Triangle` accepts a calendar development date. Apply the project convention `development_date = accident_period + development_period months`; therefore `dev_0` is observed in its accident period.

Produces: `package_input`, a three-column long DataFrame for `chainladder`.

In [5]:
origin_period = pd.PeriodIndex(loss_incurred_long['accident_period'].astype(str), freq='M')
package_input = loss_incurred_long.drop(columns='development_period').copy()
package_input['development_date'] = [
    str(origin + int(lag))
    for origin, lag in zip(origin_period, loss_incurred_long['development_period'])
]
package_input = package_input[['accident_period', 'development_date', 'amount']]

print('Calendar mapping created.')
print('Each development date equals accident period plus development age.')
print('\nPackage input:')
package_input.head(10).style.format({'amount': AMOUNT_FORMAT})

Calendar mapping created.
Each development date equals accident period plus development age.

Package input:


,accident_period,development_date,amount
14520,2016-01,2016-01,"153,701"
14521,2016-02,2016-02,"174,526"
14522,2016-03,2016-03,"139,978"
14523,2016-04,2016-04,"179,441"
14524,2016-05,2016-05,"185,553"
14525,2016-06,2016-06,"257,005"
14526,2016-07,2016-07,"221,999"
14527,2016-08,2016-08,"208,672"
14528,2016-09,2016-09,"278,668"
14529,2016-10,2016-10,"188,245"


### Step 4 — Build a `chainladder.Triangle`

Purpose: convert the long DataFrame into a cumulative triangle object. `origin` identifies accident periods and `development` identifies their calendar development dates.

Produces: `triangle`, a `chainladder.Triangle` object.

In [6]:
triangle = cl.Triangle(
    package_input,
    origin='accident_period',
    development='development_date',
    columns='amount',
    cumulative=True,
    origin_format='%Y-%m',
    development_format='%Y-%m',
)

print('Package triangle shape (index, columns, origin, development):', triangle.shape)
print('Cumulative input:', triangle.is_cumulative)
print('\nCumulative triangle as constructed by the public package:')
triangle.to_frame().style.format(AMOUNT_FORMAT)

Package triangle shape (index, columns, origin, development): (1, 1, 120, 120)
Cumulative input: True

Cumulative triangle as constructed by the public package:


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120
2016-01-01 00:00:00,"153,701","168,265","205,478","184,336","201,249","196,453","176,881","200,214","187,579","190,076","199,451","177,419","222,198","195,291","223,059","212,122","235,381","206,121","213,323","220,896","189,654","224,604","231,963","225,291","219,615","208,894","222,435","218,830","210,956","206,344","202,214","194,385","213,174","186,250","198,452","201,310","205,109","210,133","203,526","194,213","199,400","186,763","201,118","191,035","187,336","186,586","188,481","185,877","197,665","182,471","184,789","183,730","182,693","181,763","178,638","177,272","175,283","172,137","177,120","178,709","172,498","172,484","173,285","172,857","169,933","178,468","175,604","172,232","175,153","173,065","173,712","175,300","176,890","178,388","178,826","173,660","173,033","174,554","174,623","172,510","172,805","176,267","178,244","173,775","172,808","173,656","172,057","173,693","175,253","174,128","177,038","174,637","175,350","173,990","177,127","171,750","173,674","171,885","176,025","176,064","173,260","177,705","173,711","171,558","173,750","175,026","173,848","178,820","175,992","177,715","177,019","171,086","173,586","169,161","173,082","176,445","172,720","173,112","174,927","175,796"
2016-02-01 00:00:00,"174,526","179,398","205,041","195,259","187,048","201,436","199,294","205,210","242,547","200,342","199,166","235,923","202,168","245,315","215,574","222,291","234,924","244,640","243,479","225,073","222,017","230,128","226,277","216,198","227,157","232,435","222,090","235,526","222,197","215,676","216,943","210,173","205,451","216,669","205,195","207,966","208,906","211,177","210,521","206,651","193,368","202,866","198,504","201,314","206,432","204,142","199,837","200,166","189,364","196,348","197,230","190,112","193,921","189,464","194,124","193,359","185,730","192,313","184,916","192,060","181,725","183,653","184,153","180,979","177,938","180,914","186,844","182,434","182,457","188,007","184,893","185,479","185,872","180,924","183,790","183,855","183,021","180,573","184,665","179,211","181,071","181,417","182,423","185,539","187,544","187,664","185,392","184,497","178,041","186,103","189,624","184,492","183,985","185,034","185,089","183,753","182,631","184,247","180,495","179,431","186,400","182,479","184,980","186,765","186,034","182,302","184,131","181,059","183,240","185,611","183,520","177,751","178,954","181,846","186,154","183,855","183,738","180,857","183,746",nan
2016-03-01 00:00:00,"139,978","194,088","215,736","175,783","217,516","183,537","202,607","199,878","206,739","236,638","226,120","201,703","174,302","233,029","233,077","225,873","217,687","253,202","206,891","219,499","210,365","232,674","221,179","252,035","224,862","220,986","228,593","214,119","210,001","224,316","213,758","225,743","218,798","215,816","214,313","212,029","214,587","207,707","196,458","208,815","204,288","197,787","209,724","201,728","193,282","201,312","197,105","191,041","199,968","192,351","199,590","191,111","198,354","194,743","192,271","187,700","193,199","190,449","186,077","188,429","181,636","180,846","184,895","185,457","186,624","185,431","189,178","181,264","186,966","185,904","189,277","185,591","186,136","190,272","187,460","186,909","184,175","185,602","180,916","183,042","186,908","180,751","185,467","185,277","186,196","181,136","180,387","187,911","185,497","186,202","189,043","185,638","180,785","184,527","187,093","184,285","184,563","183,912","188,488","187,969","181,354","189,387","183,586","189,609","182,473","187,575","184,053","184,913","187,083","183,714","185,889","185,870","187,689","188,042","189,090","185,854","186,064","18

### Step 5 — Estimate development factors

Purpose: fit `chainladder.Development`. Its default approach is a volume-weighted age-to-age development estimate using available observations. No custom selection or tail factor is introduced in this demonstration.

Produces: `developed_triangle`, the triangle with fitted development information attached.

In [7]:
development = cl.Development()
developed_triangle = development.fit_transform(triangle)

ldf_table = development.ldf_.to_frame()
cdf_table = development.cdf_.to_frame()
selection_summary = pd.DataFrame(
    {
        'Component': ['LDF estimator', 'Development periods', 'High/low exclusions', 'Tail factor'],
        'Package setting': [
            'Volume-weighted (Development default)',
            'All available observations',
            'None in this demonstration',
            'None in this demonstration',
        ],
    }
)

print('Selected/default assumptions:')
display(selection_summary)
print('Age-to-age development factors (LDFs):')
display(ldf_table.style.format(FACTOR_FORMAT))
print('Cumulative development factors (CDFs):')
cdf_table.style.format(FACTOR_FORMAT)

Selected/default assumptions:


,Component,Package setting
0,LDF estimator,Volume-weighted (Development default)
1,Development periods,All available observations
2,High/low exclusions,None in this demonstration
3,Tail factor,None in this demonstration


Age-to-age development factors (LDFs):


,1-2,2-3,3-4,4-5,5-6,6-7,7-8,8-9,9-10,10-11,11-12,12-13,13-14,14-15,15-16,16-17,17-18,18-19,19-20,20-21,21-22,22-23,23-24,24-25,25-26,26-27,27-28,28-29,29-30,30-31,31-32,32-33,33-34,34-35,35-36,36-37,37-38,38-39,39-40,40-41,41-42,42-43,43-44,44-45,45-46,46-47,47-48,48-49,49-50,50-51,51-52,52-53,53-54,54-55,55-56,56-57,57-58,58-59,59-60,60-61,61-62,62-63,63-64,64-65,65-66,66-67,67-68,68-69,69-70,70-71,71-72,72-73,73-74,74-75,75-76,76-77,77-78,78-79,79-80,80-81,81-82,82-83,83-84,84-85,85-86,86-87,87-88,88-89,89-90,90-91,91-92,92-93,93-94,94-95,95-96,96-97,97-98,98-99,99-100,100-101,101-102,102-103,103-104,104-105,105-106,106-107,107-108,108-109,109-110,110-111,111-112,112-113,113-114,114-115,115-116,116-117,117-118,118-119,119-120
(All),1.020,1.002,1.025,1.013,1.009,1.023,1.001,1.013,1.017,1.004,0.998,1.036,0.996,1.018,1.007,1.011,1.020,1.008,0.997,0.982,1.005,0.997,0.996,1.001,0.988,0.993,1.001,0.987,0.991,1.006,0.986,0.994,0.993,0.993,0.998,1.003,0.992,0.994,0.988,0.999,0.993,0.991,0.999,0.988,1.001,0.993,1.000,0.986,0.998,0.995,0.998,0.987,0.996,0.992,1.000,0.992,0.999,0.989,0.997,0.994,1.002,0.998,1.000,1.004,1.000,1.001,0.998,1.002,0.997,1.002,0.997,1.003,1.000,1.000,0.998,1.000,1.005,0.996,1.004,0.998,0.993,1.012,0.995,0.999,1.001,0.998,1.004,0.997,1.003,1.000,0.999,1.000,1.001,0.998,1.001,0.997,1.004,0.998,0.999,1.005,1.001,0.997,1.005,0.991,1.006,0.990,1.004,1.005,0.998,1.004,0.987,1.008,0.994,1.004,1.001,0.989,0.992,1.013,1.005


Cumulative development factors (CDFs):


,1-Ult,2-Ult,3-Ult,4-Ult,5-Ult,6-Ult,7-Ult,8-Ult,9-Ult,10-Ult,11-Ult,12-Ult,13-Ult,14-Ult,15-Ult,16-Ult,17-Ult,18-Ult,19-Ult,20-Ult,21-Ult,22-Ult,23-Ult,24-Ult,25-Ult,26-Ult,27-Ult,28-Ult,29-Ult,30-Ult,31-Ult,32-Ult,33-Ult,34-Ult,35-Ult,36-Ult,37-Ult,38-Ult,39-Ult,40-Ult,41-Ult,42-Ult,43-Ult,44-Ult,45-Ult,46-Ult,47-Ult,48-Ult,49-Ult,50-Ult,51-Ult,52-Ult,53-Ult,54-Ult,55-Ult,56-Ult,57-Ult,58-Ult,59-Ult,60-Ult,61-Ult,62-Ult,63-Ult,64-Ult,65-Ult,66-Ult,67-Ult,68-Ult,69-Ult,70-Ult,71-Ult,72-Ult,73-Ult,74-Ult,75-Ult,76-Ult,77-Ult,78-Ult,79-Ult,80-Ult,81-Ult,82-Ult,83-Ult,84-Ult,85-Ult,86-Ult,87-Ult,88-Ult,89-Ult,90-Ult,91-Ult,92-Ult,93-Ult,94-Ult,95-Ult,96-Ult,97-Ult,98-Ult,99-Ult,100-Ult,101-Ult,102-Ult,103-Ult,104-Ult,105-Ult,106-Ult,107-Ult,108-Ult,109-Ult,110-Ult,111-Ult,112-Ult,113-Ult,114-Ult,115-Ult,116-Ult,117-Ult,118-Ult,119-Ult
(All),0.988,0.968,0.965,0.942,0.930,0.922,0.901,0.900,0.888,0.873,0.869,0.871,0.841,0.844,0.830,0.824,0.815,0.799,0.792,0.795,0.809,0.806,0.808,0.811,0.811,0.820,0.826,0.825,0.837,0.844,0.839,0.851,0.856,0.862,0.868,0.870,0.867,0.874,0.880,0.890,0.891,0.897,0.905,0.906,0.917,0.917,0.923,0.923,0.936,0.938,0.943,0.945,0.957,0.961,0.969,0.969,0.977,0.978,0.989,0.992,0.998,0.996,0.997,0.998,0.994,0.995,0.993,0.996,0.993,0.996,0.994,0.996,0.994,0.994,0.994,0.995,0.996,0.991,0.995,0.992,0.994,1.001,0.990,0.995,0.997,0.996,0.998,0.994,0.997,0.993,0.994,0.995,0.995,0.995,0.997,0.996,0.999,0.995,0.998,0.999,0.994,0.994,0.997,0.992,1.000,0.994,1.004,1.000,0.995,0.997,0.993,1.006,0.998,1.004,1.000,0.999,1.010,1.018,1.005


### Step 6 — Project with `chainladder.Chainladder`

Purpose: fit the package's straight Chain Ladder projection to the developed triangle. The workflow is: `Triangle` → `Development` → `Chainladder`.

Produces: `chainladder_model`, containing projected ultimates and the latest observed diagonal.

In [8]:
chainladder_model = cl.Chainladder().fit(developed_triangle)
projected_triangle = chainladder_model.full_triangle_.to_frame()

print('Chain Ladder projection fitted.')
print('Projected accident periods:', len(chainladder_model.X_.odims))
print('Full triangle shape after package projection:', projected_triangle.shape)
print('\nLatest projected rows of the completed triangle:')
projected_triangle.tail(8).style.format(AMOUNT_FORMAT)

Chain Ladder projection fitted.
Projected accident periods: 120
Full triangle shape after package projection: (120, 133)

Latest projected rows of the completed triangle:


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,9999
2025-05-01 00:00:00,"399,585","393,539","379,106","490,494","466,122","397,824","470,778","469,677","475,813","483,979","486,118","485,372","502,616","500,465","509,404","512,907","518,790","529,070","533,236","531,610","522,125","524,486","523,006","520,863","521,168","515,035","511,463","511,946","505,057","500,600","503,828","496,564","493,771","490,093","486,878","485,956","487,329","483,477","480,396","474,613","474,363","471,275","466,983","466,379","460,586","460,930","457,730","457,623","451,270","450,290","448,140","447,291","441,621","439,771","436,143","436,286","432,686","432,214","427,270","425,944","423,526","424,298","423,656","423,505","425,033","424,840","425,351","424,452","425,347","424,170","425,186","424,060","425,146","425,119","425,234","424,565","424,388","426,356","424,543","426,173","425,154","421,971","426,825","424,579","424,034","424,278","423,242","425,059","423,993","425,371","425,162","424,746","424,535","424,826","423,925","424,446","423,132","424,650","423,590","422,987","424,944","425,291","423,861","426,085","422,459","425,165","420,958","422,685","424,601","423,878","425,683","420,156","423,326","420,740","422,565","423,020","418,388","414,955","420,469","422,559","422,559","422,559","422,559","422,559","422,559","422,559","422,559","422,559","422,559","422,559","422,559","422,559","422,559"
2025-06-01 00:00:00,"521,029","457,939","493,745","530,972","485,993","453,306","401,807","402,360","407,617","414,612","416,445","415,805","430,578","428,735","436,393","439,394","444,433","453,240","456,809","455,416","447,290","449,313","448,045","446,209","446,471","441,217","438,157","438,571","432,669","428,851","431,616","425,393","423,001","419,849","417,095","416,305","417,482","414,182","411,542","406,589","406,374","403,729","400,052","399,534","394,572","394,866","392,125","392,033","386,591","385,752","383,909","383,182","378,325","376,741","373,632","373,755","370,671","370,266","366,031","364,895","362,824","363,485","362,935","362,805","364,115","363,949","364,387","363,617","364,384","363,376","364,245","363,281","364,211","364,188","364,286","363,714","363,562","365,248","363,695","365,091","364,218","361,491","365,649","363,726","363,258","363,467","362,580","364,137","363,224","364,404","364,225","363,868","363,688","363,937","363,165","363,611","362,486","363,786","362,878","362,362","364,038","364,335","363,110","365,016","361,909","364,228","360,624","362,103","363,744","363,125","364,672","359,936","362,652","360,437","362,000","362,390","358,422","355,481","360,205","361,995","361,995","361,995","361,995","361,995","361,995","361,995","361,995","361,995","361,995","361,995","361,995","361,995","361,995"
2025-07-01 00:00:00,"524,674","376,709","498,454","489,940","510,032","531,404","543,601","544,349","551,460","560,924","563,404","562,539","582,524","580,032","590,392","594,452","601,269","613,184","618,012","616,128","605,134","607,871","606,156","603,672","604,026","596,917","592,778","593,338","585,353","580,188","583,928","575,510","572,273","568,010","564,284","563,215","564,807","560,342","556,771","550,070","549,780","546,201","541,226","540,526","533,812","534,211","530,502","530,378","523,015","521,879","519,387","518,404","511,832","509,688","505,483","505,649","501,477","500,929","495,199","493,662","490,861","491,755","491,010","490,836","492,607","492,383","492,976","491,933","492,971","491,607","492,784","491,479","492,737","492,707","492,839","492,065","491,859","494,140","492,039","493,928","492,747","489,058","494,683","492,081","491,449","491

### Step 7 — Calculate aggregate actual, projected ultimate, and reserve

Purpose: retrieve the package outputs directly. Reserve is calculated as `Ultimate − Actual`, where Actual is the latest observed cumulative Loss Incurred amount.

Produces: `package_results` and an aggregate `summary`.

In [23]:
ultimate = chainladder_model.ultimate_.to_frame().iloc[:, 0]
actual = chainladder_model.X_.latest_diagonal.to_frame().iloc[:, 0]
labels = [pd.Timestamp(period).strftime('%Y-%m') for period in chainladder_model.X_.odims]

package_results = pd.DataFrame(
    {'Actual': actual.to_numpy(), 'Ultimate': ultimate.to_numpy()},
    index=pd.Index(labels, name='accident_period'),
)
package_results['Reserve'] = package_results['Ultimate'] - package_results['Actual']

if not np.isfinite(package_results[['Actual', 'Ultimate', 'Reserve']].to_numpy()).all():
    raise ValueError('The package returned a non-finite result; inspect the triangle and assumptions.')

summary = package_results[['Actual', 'Ultimate', 'Reserve']].sum().rename('Amount').to_frame()
summary.index.name = 'Measure'

print('Per-accident-period package results:')
display(package_results.style.format(AMOUNT_FORMAT))
print('Aggregate package result:')
summary.style.format(AMOUNT_FORMAT)

Per-accident-period package results:


,Actual,Ultimate,Reserve
accident_period,,,
2016-01,"175,796","175,796",0
2016-02,"183,746","184,659",913
2016-03,"184,101","187,474","3,373"
2016-04,"196,952","198,915","1,963"
2016-05,"201,165","200,946",-219
2016-06,"226,717","226,714",-3
2016-07,"229,364","230,356",992
2016-08,"235,953","235,525",-427
2016-09,"242,825","244,214","1,389"


Aggregate package result:


,Amount
Measure,
Actual,"37,949,125"
Ultimate,"35,126,709"
Reserve,"-2,822,415"


### Step 8 — Audit the reserve calculation

Purpose: make the final package calculation traceable. For every accident period, reserve must equal projected ultimate less the latest observed cumulative amount.

Produces: `reserve_audit`, an explicit calculation check.

In [24]:
reserve_audit = package_results.copy()
reserve_audit['Ultimate_minus_Actual'] = reserve_audit['Ultimate'] - reserve_audit['Actual']
reserve_audit['Difference'] = reserve_audit['Reserve'] - reserve_audit['Ultimate_minus_Actual']

if not np.allclose(reserve_audit['Difference'].fillna(0.0), 0.0):
    raise AssertionError('Reserve identity failed: Reserve must equal Ultimate minus Actual.')

print('Reserve identity passed for all available accident periods.')
reserve_audit.style.format(AMOUNT_FORMAT)

Reserve identity passed for all available accident periods.


,Actual,Ultimate,Reserve,Ultimate_minus_Actual,Difference
accident_period,,,,,
2016-01,"175,796","175,796",0,0,0
2016-02,"183,746","184,659",913,913,0
2016-03,"184,101","187,474","3,373","3,373",0
2016-04,"196,952","198,915","1,963","1,963",0
2016-05,"201,165","200,946",-219,-219,0
2016-06,"226,717","226,714",-3,-3,0
2016-07,"229,364","230,356",992,992,0
2016-08,"235,953","235,525",-427,-427,0
2016-09,"242,825","244,214","1,389","1,389",0


## Interpretation

The tables above show the complete calculation path: source database fields, calendar mapping, cumulative triangle, LDFs, CDFs, package defaults, projected triangle, and reserve identity.

In [ ]:
# la funcion debe incluir:


In [13]:
import pandas as pd
import numpy as np


def chainladder_version1(
    ruta,
    concepto="Loss Incurred",
    periodos=12,
    promedio="simple",
    excluir_min_max=False
):

    """
    Función sencilla para calcular Chain Ladder a partir de una base CSV.

    Parámetros
    ----------
    ruta : str
        Ruta del archivo CSV.

    concepto : str
        Concepto de la base de datos a la que se le desea calcular la reserva por el metodo chainladder.
        Por ejemplo: "Loss Incurred", "Loss Paid"
        

    periodos : int
        Número de períodos históricos que se utilizarán para seleccionar los factores LDF.
        
    promedio : str
        Tipo de promedio para seleccionar los factores: simple o ponderado

    excluir_min_max : bool
        Valor booleano donde si es True, elimina el LDF máximo y mínimo antes de calcular el promedio.
        
    Devuelve en forma de diccionario lo siguiente:
        - datos_filtrados
        - triangulo acumulado
        - triangulo_ldf
        - ldf_seleccionado
        - atu
        - resultados
    """

    
    #-------- 1. leemos la base
    base = pd.read_csv(ruta)

    # Columnas que contiene la base/csv
    #Al cambiar de base modificar esta parte para seleccionar las columnas que nos interesan
    columnas = [
        "concept",
        "basis",
        "amount_type",
        "accident_period",
        "development_period",
        "amount"
    ]
    # revisamos que las columnas esten
    for columna in columnas:

        if columna not in base.columns:

            raise ValueError(
                f"No se encontró la columna: {columna}"
            )

    # Revisamos que el concepto de la base exista
    conceptos_disponibles = base["concept"].unique()

    if concepto not in conceptos_disponibles:

        raise ValueError(
            f"""
            El concepto '{concepto}' no existe.
            Conceptos disponibles:
            {conceptos_disponibles}
            """
        )

    # ------ 2. filtramos la base con el concepto seleccionado

    datos = base[
        (base["concept"] == concepto)
        & (base["basis"] == "month")
        & (base["amount_type"] == "cumulative")
    ].copy()

    if datos.empty:

        raise ValueError(
            "No se encontraron registros con los filtros seleccionados."
        )

    # ------- 3. Creamos el triangulo acumulado

    triangulo = datos.pivot(index="accident_period",columns="development_period",values="amount")

    # Ordenamos filas y columnas
    triangulo = triangulo.sort_index()

    triangulo = triangulo.reindex(sorted(triangulo.columns),axis=1)

    #------ 4. Calculamos triangulo ldf

    ldf = pd.DataFrame(index=triangulo.index)

    for j in range(triangulo.shape[1] - 1):

        edad_actual = triangulo.columns[j]
        edad_siguiente = triangulo.columns[j + 1]

        nombre = (f"{edad_actual}-{edad_siguiente}")

        ldf[nombre] = (triangulo.iloc[:, j + 1]/ triangulo.iloc[:, j])

    # Caso cuando el denominador es 0
    ldf = ldf.replace([np.inf, -np.inf],np.nan)

    # ------ 5. seleccionamos factores ldf

    factores_seleccionados = {}

    periodos_utilizados = {}

    for j, columna in enumerate(ldf.columns):

        factores = (ldf[columna].dropna().tail(periodos))

        # Excluimos maximos y minimos por el tema de outliers

        if excluir_min_max and len(factores) > 2:

            indice_min = factores.idxmin()
            indice_max = factores.idxmax()
            factores = factores.drop(
                index=list(set([indice_min,indice_max])))

        # Calculamos promedio simple

        if promedio == "simple":
            factor = factores.mean()

        # calculamos promedio ponderado
        elif promedio == "ponderado":

            indices = factores.index

            anterior = triangulo.loc[indices,triangulo.columns[j]]

            siguiente = triangulo.loc[indices,triangulo.columns[j + 1]]

            factor = (siguiente.sum()/ anterior.sum())

        else:

            raise ValueError(
                "El promedio debe ser 'simple' o 'ponderado'."
            )

        factores_seleccionados[columna] = factor

        periodos_utilizados[columna] = list(factores.index)

    factores_seleccionados = pd.Series(
        factores_seleccionados,name="LDF seleccionado")

    # ------ 6. factores ATU

    atu = pd.Series(index=triangulo.columns,dtype=float,name="ATU")

    # Hacemos que el ultimo factor sea 1
    atu.iloc[-1] = 1

    producto = 1

    for j in range(
        len(factores_seleccionados) - 1,-1,-1
    ):

        producto = (
            producto* factores_seleccionados.iloc[j])

        atu.iloc[j] = producto

    # ------ 7. Ultimate y reserva

    resultados = []

    for periodo_accidente, fila in triangulo.iterrows():

        valores_disponibles = fila.dropna()

        if len(valores_disponibles) == 0:
            continue

        ultimo_valor = (valores_disponibles.iloc[-1])

        actual = (valores_disponibles.index[-1])

        factor_atu = atu.loc[actual]

        ultimate = (ultimo_valor* factor_atu)

        reserva = (ultimate - ultimo_valor)

        resultados.append(
            {
                "accident_period":
                    periodo_accidente,

                "ultimo_observado":
                    ultimo_valor,

                "development_period":
                    actual,

                "ATU":
                    factor_atu,

                "ultimate":
                    ultimate,

                "reserve":
                    reserva
            }
        )

    resultados = pd.DataFrame(resultados)

    if not resultados.empty:

        resultados = resultados.set_index(
            "accident_period"
        )

    # ------ 8. Resultados

    return {
        "datos_filtrados":datos,
        "triangulo":triangulo,
        "triangulo_ldf":ldf,
        "ldf_seleccionado":factores_seleccionados,
        "atu":atu,
        "resultados":resultados,
        "periodos_usados":periodos_utilizados
    }

In [14]:
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ruta_csv = project_root /'generated_monthly_triangles_database.csv'

prueba1 =  chainladder_version1(
    ruta=ruta_csv,
    concepto="Loss Incurred",
    periodos=12,
    promedio="ponderado"
)

/var/folders/k1/_jflqvn90v31fwxjw3lf6y680000gn/T/ipykernel_11889/626884551.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ldf[nombre] = (triangulo.iloc[:, j + 1]/ triangulo.iloc[:, j])
/var/folders/k1/_jflqvn90v31fwxjw3lf6y680000gn/T/ipykernel_11889/626884551.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ldf[nombre] = (triangulo.iloc[:, j + 1]/ triangulo.iloc[:, j])
/var/folders/k1/_jflqvn90v31fwxjw3lf6y680000gn/T/ipykernel_11889/626884551.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usuall

In [15]:
prueba1["triangulo"]

development_period,0,1,2,3,4,5,6,7,8,9,...,110,111,112,113,114,115,116,117,118,119
accident_period,,,,,,,,,,,,,,,,,,,,,
2016-01,153700.711642,168265.054672,205478.444379,184336.168001,201248.543480,196453.059025,176881.337833,200214.346199,187578.698914,190075.991440,...,177018.978866,171085.713030,173585.908595,169160.747063,173081.527124,176445.361233,172720.085632,173112.466318,174926.788130,175796.196292
2016-02,174526.085249,179397.620865,205041.059805,195258.548978,187048.160649,201436.435289,199293.923502,205210.386921,242546.688042,200341.751962,...,183520.470365,177750.670575,178954.148728,181846.011293,186153.778295,183854.533150,183737.684382,180857.272493,183746.248505,NaN
2016-03,139977.733603,194087.931093,215736.346315,175783.066907,217515.571324,183537.102418,202607.299157,199878.253026,206739.049289,236638.405603,...,185889.013660,185870.257200,187689.323174,188042.155009,189089.878262,185853.845022,186063.868425,184100.501488,NaN,NaN
2016-04,179441.145220,171328.983083,197378.848951,208702.402696,208493.696364,223233.698165,187690.613593,253992.426802,225375.117705,243903.465956,...,200265.915955,199299.153969,197234.175592,195849.823081,193656.903165,201506.058677,196951.796202,NaN,NaN,NaN
2016-05,185553.143033,216977.705207,209628.471276,214214.723923,232801.660391,210101.993900,201936.310336,223423.445555,230653.011303,211583.631794,...,209927.848694,207676.902404,209507.332802,209279.782933,205822.853384,201165.375549,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08,362141.012374,432661.332612,445636.521558,541592.036871,378209.883552,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09,367005.573515,415446.207539,433412.965713,518520.051557,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10,345630.285842,346963.701868,393892.055949,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
prueba1["triangulo_ldf"]

,0-1,1-2,2-3,3-4,4-5,5-6,6-7,7-8,8-9,9-10,...,109-110,110-111,111-112,112-113,113-114,114-115,115-116,116-117,117-118,118-119
accident_period,,,,,,,,,,,,,,,,,,,,,
2016-01,1.094758,1.221159,0.897107,1.091747,0.976171,0.900375,1.131913,0.936889,1.013313,1.049322,...,0.996083,0.966482,1.014614,0.974507,1.023178,1.019435,0.978887,1.002272,1.010481,1.00497
2016-02,1.027913,1.142942,0.952290,0.957951,1.076923,0.989364,1.029687,1.181942,0.825993,0.994134,...,0.988735,0.968560,1.006771,1.016160,1.023689,0.987649,0.999364,0.984323,1.015974,NaN
2016-03,1.386563,1.111539,0.814805,1.237409,0.843788,1.103904,0.986530,1.034325,1.144624,0.955552,...,1.011841,0.999899,1.009787,1.001880,1.005572,0.982886,1.001130,0.989448,NaN,NaN
2016-04,0.954792,1.152046,1.057370,0.999000,1.070698,0.840781,1.353251,0.887330,1.082211,0.948795,...,1.008606,0.995173,0.989639,0.992981,0.988803,1.040531,0.977399,NaN,NaN,NaN
2016-05,1.169356,0.966129,1.021878,1.086768,0.902494,0.961135,1.106406,1.032358,0.917324,1.014616,...,1.000932,0.989278,1.008814,0.998914,0.983482,0.977371,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08,1.194732,1.029989,1.215322,0.698330,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09,1.131989,1.043247,1.196365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10,1.003858,1.135254,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
prueba1["atu"]

development_period
0      0.944767
1      1.005965
2      0.932445
3      0.868258
4      0.901008
         ...   
115    0.998910
116    1.009968
117    1.018323
118    1.004970
119    1.000000
Name: ATU, Length: 120, dtype: float64

In [18]:
prueba1["resultados"]

,ultimo_observado,development_period,ATU,ultimate,reserve
accident_period,,,,,
2016-01,175796.196292,119,1.000000,175796.196292,0.000000
2016-02,183746.248505,118,1.004970,184659.490494,913.241990
2016-03,184100.501488,117,1.018323,187473.858428,3373.356940
2016-04,196951.796202,116,1.009968,198915.030251,1963.234049
2016-05,201165.375549,115,0.998910,200946.031718,-219.343830
...,...,...,...,...,...
2025-08,378209.883552,4,0.901008,340769.954897,-37439.928655
2025-09,518520.051557,3,0.868258,450209.418643,-68310.632914
2025-10,393892.055949,2,0.932445,367282.832222,-26609.223727
